In [25]:
"""
Генератор простой мелодии
Создаёт WAV-файл из последовательности нот, используя синусоидальные волны.
"""

import numpy as np
import wave
from flangerfunc import flanger
from reverb import schroeder_reverb 

# Словарь: нота -> частота в герцах (нотация научной высоты тона)
NOTE_FREQUENCIES = {
    'C4': 261.63,
    'D4': 293.66,
    'E4': 329.63,
    'F4': 349.23,
    'G4': 392.00,
    'A4': 440.00,
    'B4': 493.88,
    'C5': 523.25,
}

In [26]:
SAMPLE_RATE = 44100  # стандартная частота дискретизации (сэмплов в секунду)


def generate_tone(frequency, duration, sample_rate=SAMPLE_RATE, amplitude=0.3):
    """
    Генерирует один тон заданной частоты и длительности.
    Возвращает numpy-массив с сэмплами звуковой волны.
    """
    # Массив временных точек: от 0 до duration, с шагом 1/sample_rate
    t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

    # Формула синусоиды: A * sin(2*pi*f*t)
    wave_data = amplitude * np.sin(2 * np.pi * frequency * t)

    # Плавное затухание в конце, чтобы не было щелчка на стыке нот
    fade_samples = int(sample_rate * 0.01)  # 10 мс на затухание
    fade_out = np.linspace(1, 0, fade_samples)
    wave_data[-fade_samples:] *= fade_out

    return wave_data

In [27]:
def generate_melody(notes, note_duration=0.4):
    """
    Принимает список названий нот и собирает из них мелодию.
    Возвращает единый numpy-массив со всей мелодией.
    """
    melody = np.array([], dtype=np.float32)

    for note in notes:
        frequency = NOTE_FREQUENCIES[note]
        tone = generate_tone(frequency, note_duration)
        melody = np.concatenate([melody, tone])

    return melody

In [28]:
def save_wav(filename, audio_data, sample_rate=SAMPLE_RATE):
    """
    Сохраняет numpy-массив с аудио в виде WAV-файла.
    """
    # Переводим значения из диапазона [-1, 1] в 16-битный целочисленный формат
    audio_int16 = np.int16(audio_data * 32767)

    with wave.open(filename, 'w') as wav_file:
        wav_file.setnchannels(1)          # моно
        wav_file.setsampwidth(2)          # 2 байта = 16 бит
        wav_file.setframerate(sample_rate)
        wav_file.writeframes(audio_int16.tobytes())


if __name__ == "__main__":
    # Простая мелодия — гамма вверх и вниз
    melody_notes = ['C4', 'G4', 'E4', 'C4', 'B4', 'F4', 'B4', 'C5',
                     'C5', 'B4', 'D4', 'A4', 'D4', 'E4', 'A4', 'C4']

    print("Генерируем мелодию...")
    melody = generate_melody(melody_notes)
    #melody = flanger(melody, sample_rate=SAMPLE_RATE,
    #        rate_hz=0.5, depth_ms=5.0, base_delay_ms=3.0,
     #       mix=0.8, feedback=0.2)
    melody=schroeder_reverb(melody,sample_rate=SAMPLE_RATE,mix=1)

    output_file = "melody_flanged_reverbed_crazy.wav"
    save_wav(output_file, melody)

    print(f"Готово! Мелодия сохранена в файл: {output_file}")
    print(f"Длительность: {len(melody) / SAMPLE_RATE:.2f} секунд")

Генерируем мелодию...
Готово! Мелодия сохранена в файл: melody_flanged_reverbed_crazy.wav
Длительность: 6.40 секунд
